# JPEG conversion

This script is used to convert the dcm files into JPEG images, that can be then used for training the deep learning model.

First step is conversion, subsequently the preprocessing should be applied before training the final model

## Connect to drive

In [ ]:
pip install pydicom Pillow

In [ ]:
import numpy as np
import sys
import pandas as pd
from collections import Counter, defaultdict
from IPython.display import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense,Flatten,Input,Lambda,GlobalAveragePooling2D,Dropout,Conv2D,BatchNormalization,MaxPooling2D,Input,Activation
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img,img_to_array,array_to_img
from tensorflow.keras.optimizers import Adam,RMSprop,SGD
from tensorflow.keras.applications.densenet import DenseNet169, preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping,ReduceLROnPlateau
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.decomposition import FastICA, PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import load_model
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
import os
import math
from tensorflow.keras.regularizers import l2
import json
import pickle
import random
import os
import pydicom
from PIL import Image
import numpy as np

## Conversion DICOM into JPEG
chnage the path to input_folder and output_folder to match:
- Input folder: the folder that contains you dcm data
- Output fodler: the folder where you want your JPEG data to be saved


In [ ]:
def dicom_to_jpg(input_folder, output_folder):
    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # List all DICOM files in the input folder
    dicom_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".dcm")]

    for dicom_file in dicom_files:
        # Read the DICOM file
        dcm = pydicom.dcmread(os.path.join(input_folder, dicom_file), force=True)

        # Access the Transfer Syntax UID from the main dataset
        #transfer_syntax_uid = dcm.SOPClassUID
        #sop_class_uid = dcm.SOPClassUID
        #print(dicom_file)
        # Convert the pixel data to an 8-bit image
        pixel_array = dcm.pixel_array
        # Normalize to 8-bit range (0-255)
        pixel_array = ((pixel_array - np.min(pixel_array)) / (np.max(pixel_array) - np.min(pixel_array) + 1e-5) * 255).astype(np.uint8)

        # Create a Pillow image from the 8-bit pixel data
        image = Image.fromarray(pixel_array)

        # Save the JPEG image to the output folder
        output_file = os.path.splitext(dicom_file)[0] + ".jpg"
        image.save(os.path.join(output_folder, output_file), "JPEG")


In [ ]:
def dicom_to_jpg(input_folder, output_folder):
    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # List all DICOM files in the input folder
    dicom_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".dcm")]

    for dicom_file in dicom_files:
        try:
            dicom_path = os.path.join(input_folder, dicom_file)
            dcm = pydicom.dcmread(dicom_path, force=True)

            # Check if pixel data is present
            if 'PixelData' not in dcm:
                print(f"[SKIP] No pixel data in {dicom_file}")
                continue

            # Try to extract the pixel array
            pixel_array = dcm.pixel_array  

            # Normalize to 8-bit range (0–255)
            pixel_array = ((pixel_array - np.min(pixel_array)) / 
                           (np.max(pixel_array) - np.min(pixel_array) + 1e-5) * 255).astype(np.uint8)

            # Convert to PIL image
            image = Image.fromarray(pixel_array)

            # Save as JPEG
            output_file = os.path.splitext(dicom_file)[0] + ".jpg"
            image.save(os.path.join(output_folder, output_file), "JPEG")

            print(f"[OK] Converted: {dicom_file}")

        except Exception as e:
            print(f"[ERROR] {dicom_file} - {str(e)}")


In [ ]:
input_folder = ""
output_folder =''
dicom_to_jpg(input_folder, output_folder)